# Example 1 - Artificial Neural Network with Decoupled Neural Interfaces
**By Epifanios Baikas**

<a href="https://colab.research.google.com/github/EDGYOrganism/dni_modules/blob/main/jupyter/ann_dni.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This example demonstrates how to build an Artificial Neural Network using the components of the [<code>dni_modules</code>](../dni_modules.rst) package and train it on the MNIST dataset.

## 1. Enviroment setup

### 1.1 Download and install dependencies

In [ ]:
!git clone https://github.com/EDGYOrganism/dni_modules.git
%cd dni_modules
!pip install .

### 1.2 Import packages

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from snntorch import functional as SF

import sys
import time 
from tqdm import tqdm
from pathlib import Path

from dni_modules import ReLUActivationBuilder, LinearDNIBuilder, LinearSynthGradBuilder, LinearDecoupledNet

### 1.3 Define auxiliary functions

In [2]:
def collate_time_first(batch):
    """Custom collate function to return [T, B, ...] instead of [B, T, ...]"""
    data, labels = zip(*batch)
    data = torch.stack(data)  # [B, T, ...]
    labels = torch.stack(labels)  # [B]
    data = data.permute(1, 0, *range(2, data.ndim))  # [B, T, ...] -> [T, B, ...]
    return data, labels


def apply_spike_coding(data_loader, num_steps, batch_size, coding_type):
    # Apply either rate or temporal coding
    labels = []
    spk_data_list = []
    for data, targets in iter(data_loader):
        if coding_type == "rate":
            spk_data = spikegen.rate(data, num_steps=num_steps)
        elif coding_type == "temporal":
            spk_data = spikegen.latency(data, num_steps=num_steps)
        elif coding_type == "no_coding":
            spk_data = data.unsqueeze(0)  # Add temporal dimension
        else:
            raise RuntimeError(
                f"Invalid spike coding type: coding_type = {coding_type}"
            )

        # Replace time with batch dimension
        spk_data = spk_data.permute(
            1, 0, *range(2, spk_data.ndim)
        )  # [T, B, ...] -> [B, T, ...]

        spk_data_list.append(spk_data)
        labels.append(targets)

    # Concatenate all batches
    spk_data_tensor = torch.cat(spk_data_list)  # [B, T, ...]
    labels = torch.cat(labels)  # [B]

    # Create a new dataset
    processed_dataset = TensorDataset(spk_data_tensor, labels)

    # New dataloader with custom collate function to return [T, B, ...] format
    processed_data_loader = DataLoader(
        processed_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_time_first,
    )
    return processed_data_loader


def iterate_dataloader(
    net,
    dataloader,
    lr,
    local_update,
    loss_fn,
    optimizer,
    device,
    gain=1.0,
    is_training=True,
    compute_accuracy=True,
):
    """
    Iterate over a dataloader for training or evaluation.

    Returns:
        tuple: (epoch_loss, total_examples, correct_pred)
    """

    epoch_loss = 0
    total_examples = 0
    correct_pred = 0

    net.train() if is_training else net.eval()
    for data, targets in tqdm(
        iter(dataloader), ascii=True, ncols=80, disable=not sys.stderr.isatty()
    ):
        data = data.to(device)
        data = gain * data
        targets = targets.to(device)

        # Zero gradients
        optimizer.zero_grad()

        # Forward pass
        out = net(data, targets, lr, loss_fn, local_update)

        # Turn targets into one-hot vectors
        # targets_one_hot -> [B, num_classes]
        # out -> [T, B, num_classes]
        targets_one_hot = nn.functional.one_hot(
            targets, num_classes=out.shape[2]
        ).float()
        loss_val = loss_fn(out, targets_one_hot.unsqueeze(0))

        # Update parameters using optimizer
        if is_training:
            optimizer.step()

        # Update total epoch loss (loss_val is the average loss of the minibatch, hence it is multiplied by the batch size)
        T = out.shape[0]
        B = out.shape[1]
        epoch_loss += B * loss_val.item()
        total_examples += targets.size(0)

        if compute_accuracy:
            if T > 1:
                correct_pred += SF.acc.accuracy_rate(out, targets) * targets.size(0)
            else:
                predicted_labels = torch.argmax(out[0], dim=1)
                correct_pred += (predicted_labels == targets).sum().item()

    return (
        epoch_loss,
        total_examples,
        correct_pred,
    )


def squared_error_loss(x, y):
    """Squared error loss function"""
    return 0.5 * torch.sum((x - y) ** 2)

### 1.4 Set random seeds for reproducibility 

In [3]:
random_seed = 42
torch.manual_seed(random_seed)

### 1.5 Load MNIST dataset


In [4]:
# Dataloader arguments
num_steps = 1
batch_size = 256
data_path = str(Path.cwd().parent / 'data')

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

# Define transform
transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])

mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)

# Create Dataloaders
train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, drop_last=False)
test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=True, drop_last=False)

spk_train_loader = apply_spike_coding(train_loader, num_steps, batch_size, coding_type="no_coding")
spk_test_loader = apply_spike_coding(test_loader, num_steps, batch_size, coding_type="no_coding")

## 2. DNI network setup

With all the necessary packages in place, let's build a feed-forward ANN with two hidden DNIs. To achieve this, we first need to create the necessary builder objects that will be used for instantiating each part of the ANN.

### 2.1 DNIBuilder and ActivationBuilder

For this example, we want each DNI to consist of a linear layer without a bias vector, followed by a batch normalization layer and a ReLU activation. We begin by creating a [<code>ReLUActivationBuilder</code>](../dni_modules.rst#dni_modules.activation_builder.ReLUActivationBuilder) and a [<code>LinearDNIBuilder</code>](../dni_modules.rst#dni_modules.dni_builder.LinearDNIBuilder) object. We initially set the <code>in_features</code> and <code>out_features</code> parameters of the <code>LinearDNIBuilder</code> object to 1, as they will be later modified when the object is used for building DNIs.

In [5]:
activation_builder = ReLUActivationBuilder()

dni_builder = LinearDNIBuilder(in_features=1,
                              out_features=1,
                              bias=True,
                              activation_builder=activation_builder,
                              batch_norm=True)

### 2.2 SynthGradBuilder

We also need to create a builder object for synthetic gradient modules. Since we are using Linear DNIs, we need to instantiate a [<code>LinearSynthGradBuilder</code>](../dni_modules.rst#dni_modules.synth_grad_builder.LinearSynthGradBuilder). For this example, we create synthetic gradient modules with 2 hidden linear layers. The output dimension of each hidden linear layer is controlled by the <code>hidden_layer_size</code> parameter. By default, batch normalization is included in-between the hidden linear layer and its activation. We can initialize the parameters of the output layer to zero by setting the <code>output_zero_init</code> parameter to <code>True</code>.

In [6]:
synth_grad_builder = LinearSynthGradBuilder(in_features=1,
                                            out_features=1,
                                            hidden_layer_size=1024,
                                            bias=False,
                                            activation_builder=activation_builder,
                                            num_hidden=2,
                                            output_zero_init=True)

### 2.3 LinearDecoupledNet

We can now pass the builder objects that we defined to the constructor of the [<code>LinearDecoupledNet</code>](../dni_modules.rst#dni_modules.decoupled_net.LinearDecoupledNet) class and build the complete network. 

In [7]:
net = LinearDecoupledNet(in_features=784,
                        hidden_dni_features=256,
                        out_features=10,
                        dni_builder=dni_builder,
                        synth_grad_builder=synth_grad_builder,
                        num_hidden_dni=2
                    ).to(device)

## 3. Train network

### 3.1 Set loss function and optimizer

In [8]:
loss_fn = squared_error_loss
scheduler = None

lr = 3e-5
gain = 1.0
local_update = False
num_epochs = 50
optimizer = torch.optim.Adam(net.parameters(), lr=lr)

### 3.2 Training loop

In [9]:
# Training loop ---------------------------------------------------------------------------------------------------------------------------------------
epoch_train_loss_hist = []
epoch_test_loss_hist = []

epoch_train_acc_hist = []
epoch_test_acc_hist = []

max_test_acc = 0

# Outer training loop
for epoch in range(num_epochs):
    epoch_start_time = time.time()

    epoch_train_loss = 0
    epoch_test_loss = 0

    # Train network on spk_train_loader
    (
        _,
        _,
        _,
    ) = iterate_dataloader(
        net=net,
        dataloader=spk_train_loader,
        lr=lr,
        local_update=local_update,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
        gain=gain,
        is_training=True,
        compute_accuracy=False,
    )

    # Evaluate loss and accuracy on spk_train_loader with the final trained parameters of the epoch
    epoch_train_loss, total_train, correct_train = iterate_dataloader(
        net=net,
        dataloader=spk_train_loader,
        lr=lr,
        local_update=local_update,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
        gain=gain,
        is_training=False,
        compute_accuracy=True,
    )

    train_acc = correct_train / total_train

    # Test network on spk_test_loader
    epoch_test_loss, total_test, correct_test = iterate_dataloader(
        net=net,
        dataloader=spk_test_loader,
        lr=lr,
        local_update=local_update,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
        gain=gain,
        is_training=False,
        compute_accuracy=True,
    )
    test_acc = correct_test / total_test

    if test_acc > max_test_acc:
        best_net_param = {k: v.clone() for k, v in net.state_dict().items()}
        max_test_acc = test_acc

    epoch_train_loss_hist.append(epoch_train_loss / total_train)
    epoch_test_loss_hist.append(epoch_test_loss / total_test)
    epoch_train_acc_hist.append(train_acc)
    epoch_test_acc_hist.append(test_acc)

    epoch_time = time.time() - epoch_start_time

    if scheduler is not None:
        scheduler.step()

    print(
        f"Epoch: {epoch + 1}, Train loss = {epoch_train_loss_hist[epoch]:.4f}, Test loss = {epoch_test_loss_hist[epoch]:.4f}, Train acc = {train_acc:.4f}, Test acc = {test_acc:.4f}, Epoch time = {epoch_time:.2f}s"
    )

Epoch: 1, Train loss = 42.6027, Test loss = 42.2112, Train acc = 0.8196, Test acc = 0.8252, Epoch time = 2.90s
Epoch: 2, Train loss = 26.5695, Test loss = 26.2313, Train acc = 0.8997, Test acc = 0.9042, Epoch time = 1.47s
Epoch: 3, Train loss = 21.4054, Test loss = 21.2240, Train acc = 0.9227, Test acc = 0.9244, Epoch time = 1.53s
Epoch: 4, Train loss = 18.5355, Test loss = 18.5123, Train acc = 0.9348, Test acc = 0.9344, Epoch time = 1.46s
Epoch: 5, Train loss = 16.4542, Test loss = 16.5966, Train acc = 0.9425, Test acc = 0.9418, Epoch time = 1.47s
Epoch: 6, Train loss = 14.8843, Test loss = 15.0921, Train acc = 0.9480, Test acc = 0.9477, Epoch time = 1.56s
Epoch: 7, Train loss = 13.7644, Test loss = 14.0573, Train acc = 0.9523, Test acc = 0.9509, Epoch time = 1.45s
Epoch: 8, Train loss = 12.8680, Test loss = 13.2070, Train acc = 0.9556, Test acc = 0.9531, Epoch time = 1.52s
Epoch: 9, Train loss = 12.1745, Test loss = 12.5924, Train acc = 0.9585, Test acc = 0.9556, Epoch time = 1.45s
E